In [ ]:
!pip install snowballstemmer

In [1]:
import pandas as pd
import spacy
import snowballstemmer

In [2]:
# 1. Setup

nlp = spacy.load("en_core_web_sm")
stemmer = snowballstemmer.stemmer("english")


def preprocess(text: str) -> dict:
    """Run the full pipeline on one piece of text: sentence segmentation,
    tokenization, lowercasing, stopword removal, lemmatization, stemming."""
    if not isinstance(text, str) or not text.strip():
        return {"sentences": [], "tokens": [], "clean_tokens": [], "lemmas": [], "stems": []}

    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]
    tokens = [tok.text for tok in doc]

    # Stopword removal (keep only alphabetic, non-stopword tokens)
    clean_tokens_docs = [tok for tok in doc if tok.is_alpha and not tok.is_stop]
    clean_words = [tok.lower_ for tok in clean_tokens_docs]

    # Lemmatization (spaCy is POS-aware automatically)
    lemmas = [tok.lemma_.lower() for tok in clean_tokens_docs]

    # Stemming (for comparison only)
    stems = stemmer.stemWords(clean_words)

    return {
        "sentences": sentences,
        "tokens": tokens,
        "clean_tokens": clean_words,
        "lemmas": lemmas,
        "stems": stems,
    }

In [3]:
# 2. Load YOUR dataset

df = pd.read_csv(r"C:/Users/HP/Desktop/Internship Final Submission/NLP/2_raw_jobs.csv")
print(df.columns.tolist())
print(df.shape)

# 3. Apply preprocessing to every job description

results = df["job_description"].apply(preprocess)

df["clean_tokens"] = results.apply(lambda r: r["clean_tokens"])
df["lemmas"] = results.apply(lambda r: r["lemmas"])
df["clean_text"] = df["lemmas"].apply(lambda toks: " ".join(toks))

# Optional: keep stems too, useful for the stemming-vs-lemma comparison
df["stems"] = results.apply(lambda r: r["stems"])



# 4. Peek at the result

print(df[["job_id", "job_title", "clean_text"]].head())


# 5. Save the preprocessed dataset

output_path = r"C:/Users/HP/Desktop/Internship Final Submission/NLP/job_dataset_preprocessed.csv"
df.to_csv(output_path, index=False)
print(f"\nSaved preprocessed dataset to: {output_path}")
print(f"Rows: {df.shape[0]}")

['job_id', 'job_title', 'company', 'location', 'job_description', 'experience', 'education', 'salary', 'job_type']
(1000, 9)
   job_id                  job_title  \
0  100001   Supply Chain Coordinator   
1  100002  Backend Software Engineer   
2  100003             UX/UI Designer   
3  100004             Data Scientist   
4  100005             Data Scientist   

                                          clean_text  
0  look motivated supply chain coordinator join g...  
1  join backend software engineer help drive team...  
2  look motivated ux ui designer join grow team r...  
3  seek talented data scientist passionate delive...  
4  join data scientist help drive team success re...  

Saved preprocessed dataset to: C:/Users/HP/Desktop/Internship Final Submission/NLP/job_dataset_preprocessed.csv
Rows: 1000
